# 리포트 02 — 표적 모델: 메쉬 7종을 짓고 사진·공식 CAD·기준해로 재고, σ 의 주파수 축을 측정에 맞췄다

> ### 한 일
> **드론 7종의 메쉬를 제원과 제조사 CAD 치수에서 세워 사진 실루엣·공식 CAD 치수와 맞대고, 그 메쉬에 Sionna 광선엔진의 가림 판정과 부품별 재질 PO 면적분을 걸어 RCS 를 계산한 뒤, σ 의 주파수 의존성을 Das 측정에 맞췄다.**

### 결과
1. **기체 7 ⟨outputs/report02_derived.json : mesh.n⟩종의 메쉬를 지었다**(§1 그림 1) — Mini 5 Pro ⟨outputs/report02_derived.json : mesh.smallest⟩ 376 mm ⟨outputs/report02_derived.json : mesh.smallest_span_mm⟩ 부터 S1000+ ⟨outputs/report02_derived.json : mesh.largest⟩ 1330 mm ⟨outputs/report02_derived.json : mesh.largest_span_mm⟩ 까지 3.54 ⟨outputs/report02_derived.json : mesh.span_ratio⟩배, 부품 219 ⟨outputs/report02_derived.json : mesh.n_parts_total⟩개 · 삼각형 207010 ⟨outputs/report02_derived.json : mesh.n_tris_total⟩개, 전기적 크기 kr 8.4 ⟨outputs/report02_derived.json : electrical.kr_min⟩~79.7 ⟨outputs/report02_derived.json : electrical.kr_max⟩.
2. **사진 21 ⟨outputs/report02_derived.json : photo.n_pairs⟩장과 실루엣으로 맞댔다**(그림 2) — IoU 는 X500 V2 ⟨outputs/report02_derived.json : photo.best⟩ 0.875 ⟨outputs/report02_derived.json : photo.best_iou⟩(자기복제 상한의 91% ⟨outputs/report02_derived.json : photo.best_pct⟩)부터 Mini 5 Pro ⭐ ⟨outputs/report02_derived.json : photo.worst⟩ 0.478 ⟨outputs/report02_derived.json : photo.worst_iou⟩(50% ⟨outputs/report02_derived.json : photo.worst_pct⟩)까지, 외곽오차는 2.2 ⟨outputs/report02_derived.json : photo.contour_min_mm⟩~19.0 mm ⟨outputs/report02_derived.json : photo.contour_max_mm⟩ 다.
3. **부품별 재질과 광선 가림이 이 편의 기여다**(그림 3·4) — 도전성 carbon, metal, camera_assembly, pcb ⟨outputs/report02_derived.json : material.conducting⟩ 가 면적의 45.3% ⟨outputs/report02_derived.json : material.conducting_area_pct⟩ 를 차지하고 Σ|Γ|A 의 73.5% ⟨outputs/report02_derived.json : material.conducting_gamma_pct⟩ 를 낸다. 가림을 켜면 방위평균 σ 가 0.11 ⟨outputs/report02_derived.json : occlusion.min_db⟩(S1000+ ⟨outputs/report02_derived.json : occlusion.min_drone⟩)~6.63 dB ⟨outputs/report02_derived.json : occlusion.max_db⟩(Matrice 4E ⭐ ⟨outputs/report02_derived.json : occlusion.max_drone⟩) 내려간다.
4. 커널이 해석 PO 기준해와 kr 1 ⟨outputs/sbr_kr_sweep.json : summary_div16.kr_min⟩~100 ⟨outputs/sbr_kr_sweep.json : summary_div16.kr_max⟩ 전 구간에서 최대 0.201 dB ⟨outputs/sbr_kr_sweep.json : summary_div16.max_abs_db_vs_po⟩ 안에서 일치하고 (kr 21 ⟨outputs/sbr_kr_sweep.json : summary_div16.n_points⟩점 × 입사 48 ⟨outputs/sbr_kr_sweep.json : meta.n_incidence⟩방향), 다중반사 위상은 PEC 이면각 해석해 8πa²b²/λ² 와 변 길이 4점 전부에서 0.556 dB ⟨outputs/sbr_defect_fixes.json : d3_multibounce_phase.max_abs_err_db⟩ 안에서 맞는다.
5. **주파수 의존성 A(f) 만 Das 측정(IEEE WCL 2026 15:3731)에 맞췄다** — 기울기 0.210 dB/GHz ⟨outputs/rcs_anchor.json : literature.mu_eps.multiband_phantom3.mu_a⟩, 그때 기체 7 ⟨outputs/report02_derived.json : anchor_modes.n_airframes⟩종의 평균 레벨이동은 0.00 dB ⟨outputs/report02_derived.json : anchor_modes.level_shift_abs_max_db⟩, 정규화 각도패턴 이동은 1.9e-15 dB ⟨outputs/report02_derived.json : anchor.shape_invariance_max_abs_db⟩ 다.

### 방법

| 무엇을 | 어떻게 얻었나 |
|---|---|
| 메쉬 | 제원(공식 외형 L×W×H · 모터 대각 · 프롭 지름)과 제조사 CAD 치수표에서 세우고 부품을 재질 그룹으로 유지 (`src/drone_cad.py`, `src/drones.py:822` build_drone) |
| 형상 검사 | 사진 실루엣 IoU(자세·원근·배율·프롭위상 정합 후) · 제조사 CAD 치수 20개 · 실물 스캔 (`src/viz_mesh_photo.py`, `src/viz_cad_compare.py`) |
| 가림 | Sionna 의 Mitsuba/OptiX 광선엔진으로 면별 first-hit 판정 (`src/rcs_sbr.py:184`) |
| σ | 조명면 위 PO 표면적분, 부품별 \|Γ\| 가중 + 얇은 셸 뒤 금속의 코히런트 합 |
| 검증 | 기준해 셋과 대조 — 해석 PO 구 · 정확 Mie 구 · PEC 이면각 닫힌형 (`benchmark/mie_pec_sphere.py`, `benchmark/verify_sbr_defect_fixes.py`) |
| 절대 레벨 | 우리 PO 면적분 출력 그대로 — 재보정(`slope_only`)은 밴드평균 레벨을 축으로 σ(f) 를 회전시킨다 (`src/sigma_anchor.py:589`) |
| 주파수 의존성 | σ = A(f)·B₁(φ,θ)·B₂ 분해에서 **A(f) 의 기울기만** Das 측정으로 교체 — B₁ 은 우리 기하 계산 그대로 (`src/sigma_anchor.py`) |

### 재현

```bash
# ① 메쉬 원장 넷 — 기하·사진·공식 CAD·재질/가림 (CPU)
SIONNA2_CPU=1 PYTHONPATH=src:benchmark ~/.venvs/py312/bin/python src/viz_mesh_gallery.py
SIONNA2_CPU=1 PYTHONPATH=src:benchmark ~/.venvs/py312/bin/python src/viz_mesh_photo.py
SIONNA2_CPU=1 PYTHONPATH=src:benchmark ~/.venvs/py312/bin/python src/viz_cad_compare.py
SIONNA2_CPU=1 PYTHONPATH=src:benchmark ~/.venvs/py312/bin/python src/viz_mesh_material.py
# ② 기준해 대조와 앵커 원장 (GPU 1장)
SIONNA2_GPU=2 PYTHONPATH=src:benchmark ~/.venvs/py312/bin/python benchmark/verify_sbr_kr_sweep.py
SIONNA2_GPU=2 PYTHONPATH=src:benchmark ~/.venvs/py312/bin/python benchmark/verify_sbr_defect_fixes.py
SIONNA2_GPU=2 PYTHONPATH=src:benchmark ~/.venvs/py312/bin/python benchmark/rcs_anchor.py
PYTHONPATH=src:benchmark ~/.venvs/py312/bin/python src/sigma_anchor.py
# ③ σ 오차 → 순위 강건성 (§5) — CPU
PYTHONPATH=src:benchmark ~/.venvs/py312/bin/python benchmark/sigma_sensitivity.py
# ④ 이 리포트 재생성 (파생 JSON + 게재규격 그림 4장 + 노트북) — GPU 불필요
PYTHONPATH=src:benchmark ~/.venvs/py312/bin/python src/make_report02_target.py
```

| | |
|---|---|
| 출력 | `outputs/mesh_gallery.json`, `outputs/mesh_compare_photo.json`, `outputs/mesh_compare_cad.json`, `outputs/mesh_compare_material.json`, `outputs/sbr_kr_sweep.json`, `outputs/sbr_defect_fixes.json`, `outputs/rcs_anchor.json`, `outputs/sigma_anchor.json`, `outputs/sigma_sensitivity.json`, `outputs/report02_derived.json` |
| 소요 | 메쉬 원장 넷 33 ⟨outputs/mesh_gallery.json : _meta.runtime_s⟩ + 3038 ⟨outputs/mesh_compare_photo.json : _meta.runtime_s⟩ + 60 ⟨outputs/mesh_compare_cad.json : meta.elapsed_s⟩ + 130 s ⟨outputs/mesh_compare_material.json : _meta.runtime_s⟩ (CPU) · kr 스윕 176 s ⟨outputs/sbr_kr_sweep.json : meta.runtime_s⟩ · 앵커 11407 s ⟨outputs/rcs_anchor.json : meta.runtime_s⟩ (GPU 1장) · σ 민감도 15 s ⟨outputs/sigma_sensitivity.json : _meta.runtime_s⟩ · 리포트 빌드 20 s ⟨outputs/report02_derived.json : _meta.runtime_s⟩ |
| 비고 | 메쉬 소스 최신 편집 2026-07-31 06:07:35 ⟨outputs/report02_derived.json : _meta.mesh_source_newest⟩ 기준으로, 이 편이 읽는 JSON 15 ⟨outputs/report02_derived.json : provenance.n_sources⟩개 중 0 ⟨outputs/report02_derived.json : provenance.n_fresh⟩개가 그보다 새것이다. 나머지 15 ⟨outputs/report02_derived.json : provenance.n_stale⟩개(§4·§5)의 재실행은 §6 표 첫 두 줄이고, 아래 코드 셀이 원장을 그대로 찍는다. |

### 앞 편에서

| 어디서 | 무엇을 알고 와야 하나 |
|---|---|
| 01편 §3 | 게재된 선행이 표적 산란을 어떻게 다뤘는지 — 측정 · 피팅 · stock Fresnel · 해석적 블레이드 |

<!--pk:paper_map {"kind": "paper_map", "sections": ["III-A. Target Model"], "claim": "표적 σ 는 부품별 재질 메쉬 위에서 광선으로 조명면을 가려낸 뒤 PO 면적분으로 계산하고, 절대 레벨은 그 계산 출력으로 두되 주파수 기울기만 측정 앵커에 맞춘다.", "evidence": ["그림 5", "그림 6", "그림 7", "표 §3.1", "표 §5", "outputs/sbr_kr_sweep.json:summary_div16.max_abs_db_vs_po", "outputs/sigma_anchor.json:drones", "outputs/sigma_sensitivity.json:common_mode.order_invariant_everywhere"], "qualifications": ["공통모드 σ 오차는 세 밴드를 함께 옮겨 순위에서 상쇄된다 — 절대거리만 σ 1 dB 당 0.246 dB ⟨outputs/report02_derived.json : sigma_sens.slope_mean_db_per_db⟩ 움직인다(§5)", "밴드별 차분 σ 오차가 순위를 정하는 축이고, 자세평균 인용 + 측정 기울기 앵커가 뒤집힘 문턱을 0.09 ⟨outputs/report02_derived.json : sigma_sens.worst_flip_aspect_avg_db⟩→3.72 dB ⟨outputs/report02_derived.json : sigma_sens.worst_flip_anchored_db⟩ 로 올린다", "바이스태틱 자세 패턴은 β ≤ 45° 에서 성립한다(§2.1)", "Rzewuski(NATO STO-MP-MSG-SET-183, 2021, 게재)가 FDTD 로 드론 바이스태틱 RCS 를 패시브 예산에 이미 넣었다 — 이 편의 기여는 엔진과 파이프라인 통합이다"], "report": "report02_target"}-->
> **논문 대응** · **III-A. Target Model**
>
> 주장 — 표적 σ 는 부품별 재질 메쉬 위에서 광선으로 조명면을 가려낸 뒤 PO 면적분으로 계산하고, 절대 레벨은 그 계산 출력으로 두되 주파수 기울기만 측정 앵커에 맞춘다.
> 근거 — 그림 5 · 그림 6 · 그림 7 · 표 §3.1 · 표 §5 · `outputs/sbr_kr_sweep.json:summary_div16.max_abs_db_vs_po` · `outputs/sigma_anchor.json:drones` · `outputs/sigma_sensitivity.json:common_mode.order_invariant_everywhere`
> 단서 — 공통모드 σ 오차는 세 밴드를 함께 옮겨 순위에서 상쇄된다 — 절대거리만 σ 1 dB 당 0.246 dB ⟨outputs/report02_derived.json : sigma_sens.slope_mean_db_per_db⟩ 움직인다(§5) · 밴드별 차분 σ 오차가 순위를 정하는 축이고, 자세평균 인용 + 측정 기울기 앵커가 뒤집힘 문턱을 0.09 ⟨outputs/report02_derived.json : sigma_sens.worst_flip_aspect_avg_db⟩→3.72 dB ⟨outputs/report02_derived.json : sigma_sens.worst_flip_anchored_db⟩ 로 올린다 · 바이스태틱 자세 패턴은 β ≤ 45° 에서 성립한다(§2.1) · Rzewuski(NATO STO-MP-MSG-SET-183, 2021, 게재)가 FDTD 로 드론 바이스태틱 RCS 를 패시브 예산에 이미 넣었다 — 이 편의 기여는 엔진과 파이프라인 통합이다

---

## §1. 일곱 대의 기체 — 무엇을 지었나

메쉬는 제원과 제조사 CAD 치수에서 세운다 — 공식 외형(L×W×H)·모터 대각·프롭 지름에 맞춘 뒤 부품을 **재질 그룹**으로 나눠 유지한다(`src/drones.py:43` DroneSpec, `src/drones.py:822`). 아래 넷은 전부 **현재 메쉬**를 다시 재서 그린 것이고, **matrice4e · mini5pro 가 06편 실측 대상**이다(표에서 ⭐).

![gallery](outputs/figures/mesh_gallery_all.png)

**그림 1.** 우리가 지은 일곱 대는 어떻게 생겼고 서로 얼마나 다른가?

크기 폭이 3.54 ⟨outputs/report02_derived.json : mesh.span_ratio⟩배다 — Mini 5 Pro ⟨outputs/report02_derived.json : mesh.smallest⟩ 376 mm ⟨outputs/report02_derived.json : mesh.smallest_span_mm⟩ 대 S1000+ ⟨outputs/report02_derived.json : mesh.largest⟩ 1330 mm ⟨outputs/report02_derived.json : mesh.largest_span_mm⟩. 그 폭이 kr 을 8.4 ⟨outputs/report02_derived.json : electrical.kr_min⟩(DJI Mini 5 Pro ⟨outputs/report02_derived.json : electrical.kr_min_name⟩ @ LTE ⟨outputs/report02_derived.json : electrical.kr_min_band⟩)~79.7 ⟨outputs/report02_derived.json : electrical.kr_max⟩(DJI S1000+ ⟨outputs/report02_derived.json : electrical.kr_max_name⟩ @ WiFi ⟨outputs/report02_derived.json : electrical.kr_max_band⟩)로 벌리고, 거기가 §3 의 두 눈금이 걸리는 자리다.

| 기체 | L×W×H [mm] (프롭 포함) | 무게 [g] | 부품 | 재질 그룹 | 삼각형 | 외접반경 r [m] | kr @5G |
|---|---|---|---|---|---|---|---|
| Mini 5 Pro ⭐ | 251 × 376 × 91 | 250 | 25 | 9 | 29,044 | 0.217 | 15.9 |
| Phantom 4 | 440 × 468 × 198 | 1380 | 23 | 8 | 28,402 | 0.311 | 22.8 |
| X500 V2 | 552 × 581 × 335 | 1650 | 48 | 10 | 18,816 | 0.390 | 28.6 |
| Mavic 4 Pro | 464 × 596 × 150 | 1063 | 20 | 8 | 29,932 | 0.359 | 26.4 |
| Matrice 4E ⭐ | 470 × 599 × 150 | 1219 | 25 | 9 | 30,714 | 0.358 | 26.3 |
| Typhoon H (H480) | 634 × 704 × 319 | 1950 | 29 | 9 | 34,732 | 0.389 | 28.5 |
| S1000+ | 1330 × 1330 × 380 | 9500 | 49 | 10 | 35,370 | 0.730 | 53.6 |

출처 ⟨outputs/report02_derived.json : mesh.rows⟩

부품 수는 그룹별 연결성분의 합이다 — 모터 4개는 4로 센다. 외접반경 r 은 이 빌더가 메쉬에서 직접 다시 재고 갤러리 원장과 대조한다(최대 차이 0.00% ⟨outputs/report02_derived.json : mesh.r_crosscheck_max_pct⟩).

### §1.1 사진과 맞댔다

사진 21 ⟨outputs/report02_derived.json : photo.n_pairs⟩장을 각각 정합해 실루엣 IoU 를 쟀다 — 카메라 자세·원근·배율·위치와 로터별 프로펠러 위상을 맞춘 뒤 겹친다(`src/viz_mesh_photo.py`). 자료 규칙에 걸린 사진 12 ⟨outputs/report02_derived.json : photo.n_excluded⟩장은 사유와 함께 원장에 남겼다.

<!--pk:figure {"kind": "figure", "path": "outputs/figures/report02_f2_mesh_photo.png", "figure_no": "2", "question": "우리 메쉬의 외형은 실제 기체 사진과 얼마나 맞는가?", "paper_caption": "Silhouette overlay of each mesh against a photograph of the real airframe, with the self-replication ceiling of the same metric shown beside every score.", "vector_pdf": "outputs/figures/report02_f2_mesh_photo.pdf", "report": "report02_target"}-->
![report02_f2_mesh_photo.png](outputs/figures/report02_f2_mesh_photo.png)

**그림 2.** 우리 메쉬의 외형은 실제 기체 사진과 얼마나 맞는가?

지표에 눈금을 붙였다 — 같은 메쉬로 만든 가짜 사진을 같은 파이프라인에 넣으면 IoU 0.864 ⟨outputs/report02_derived.json : photo.ceiling_min⟩~0.957 ⟨outputs/report02_derived.json : photo.ceiling_max⟩ 가 나온다(암·블레이드가 몇 px 이라 상한이 1.0 아래다). 같은 메쉬끼리도 자세가 1° 어긋나면 0.906 ⟨outputs/report02_derived.json : photo.iou_at_1deg_pose_error⟩, 2° 면 0.822 ⟨outputs/report02_derived.json : photo.iou_at_2deg_pose_error⟩ 로 내려간다.

### §1.2 형상 검사 세 가지 — 한 표로

| 검사 | 대상 | 지표 | 값 | 그 지표의 바닥 |
|---|---|---|---|---|
| 사진 실루엣 | 7기체 · 사진 21 ⟨outputs/report02_derived.json : photo.n_pairs⟩장 | IoU (상한 대비) | 0.478 ⟨outputs/report02_derived.json : photo.worst_iou⟩~0.875 ⟨outputs/report02_derived.json : photo.best_iou⟩ (50 ⟨outputs/report02_derived.json : photo.worst_pct⟩~91% ⟨outputs/report02_derived.json : photo.best_pct⟩) | 자기복제 상한 0.864 ⟨outputs/report02_derived.json : photo.ceiling_min⟩~0.957 ⟨outputs/report02_derived.json : photo.ceiling_max⟩ · 자세 1° 오차에서 0.906 ⟨outputs/report02_derived.json : photo.iou_at_1deg_pose_error⟩ |
| 제조사 CAD 치수 | 어셈블리 3 ⟨outputs/report02_derived.json : cad.n_assemblies⟩종 · 치수 20 ⟨outputs/report02_derived.json : cad.n_dims⟩개 | 최대 편차 | CAD 대비 0.56 ⟨outputs/report02_derived.json : cad.worst_vs_cad_pct⟩ % · 발행 제원 대비 0.56% ⟨outputs/report02_derived.json : cad.worst_vs_published_pct⟩ | 제조사 CAD ↔ 자사 발행 제원이 6.90% ⟨outputs/report02_derived.json : cad.floor_pct⟩ 까지 갈린다 |
| 실물 유래 메쉬 | Phantom 4 real scan (0.4mm) ⟨outputs/phantom4_scan_compare.json : name_real⟩ 스캔 · Typhoon H480 (real CAD, Apache-2.0) ⟨outputs/real_cad_compare.json : typhoon.name_real⟩ · DJI Matrice 600 Pro (2016 hexa) (community mesh) ⟨outputs/community_compare.json : m600.name_real⟩ | Δ 방위평균 σ | -2.00 ⟨outputs/phantom4_scan_compare.json : d_sigma_db⟩ · -0.40 ⟨outputs/real_cad_compare.json : typhoon.d_sigma_db⟩ · +1.80 dB ⟨outputs/community_compare.json : m600.d_sigma_db⟩ | 자세별 RMS 6.8 ⟨outputs/phantom4_scan_compare.json : d_sigma_rms_db⟩~10.0 dB ⟨outputs/community_compare.json : m600.d_sigma_rms_db⟩ |

**이 표가 σ 의 인용 단위를 정한다** — 방위평균과 로브 위치로 인용하고, 널 깊이는 메쉬 세부에 걸리므로 자세별 RMS 열에 그 크기가 그대로 적혀 있다.

### §1.3 부품별 재질 — PO 적분에 들어가는 물리 입력

Sionna RT 와 우리 PO 적분기는 **같은 재질 표**를 읽는다(`src/materials.py:53` MATERIALS, `src/drones.py:562` DRONE_GROUP_MAT). 아래 그림은 7기체의 표면적을 재질로 나누고, 같은 면적을 |Γ| 로 가중해 **무엇이 반사 진폭을 내는지**까지 함께 싣는다.

![materials](outputs/figures/mesh_compare_material_area.png)

**그림 3.** 기체는 무엇으로 이루어져 있고, 그 중 무엇이 반사 진폭을 내는가?

도전성 carbon, metal, camera_assembly, pcb ⟨outputs/report02_derived.json : material.conducting⟩ 가 면적의 45.3% ⟨outputs/report02_derived.json : material.conducting_area_pct⟩ 를 차지하고 Σ|Γ|A 의 73.5% ⟨outputs/report02_derived.json : material.conducting_gamma_pct⟩ 를 낸다.

| 재질 | 부품 그룹 | \|Γ\| 벌크 | \|Γ\| PO 실효 | 면적 [m²] (7기체) | 면적 비중 | Σ\|Γ\|A 비중 |
|---|---|---|---|---|---|---|
| carbon | arm, deck, gear_cf | 0.989 | 0.90 | 0.558 | 17.2 % | 27.6 % |
| metal | battery, motor | 1.000 | 1.00 | 0.480 | 14.8 % | 26.3 % |
| plastic | accent, body, canopy, gear | 0.244 | 0.28 | 1.318 | 40.7 % | 20.3 % |
| camera_assembly | camera | 1.000 | 0.85 | 0.311 | 9.6 % | 14.5 % |
| prop_plastic | prop | 0.244 | 0.25 | 0.452 | 14.0 % | 6.2 % |
| pcb | fc, pcb | 1.000 | 0.80 | 0.117 | 3.6 % | 5.1 % |

출처 ⟨outputs/report02_derived.json : material.rows⟩

유전체 셸(body·canopy)은 |Γ| = 0.28 ⟨outputs/report02_derived.json : material.gamma_shell⟩ 로 왕복 투과 τ = 1−|Γ|², 즉 0.71 dB ⟨outputs/report02_derived.json : material.shell_tau_db⟩ 를 곱해 통과시키고 그 뒤 금속(배터리·PCB)을 코히런트 합산한다(`src/rcs_sbr.py:88`).

## §2. 엔진 — 광선으로 조명면을 찾고 그 위에서 PO 를 적분한다

상용 고주파 RCS 솔버(FEKO/CST SBR+)의 순서 그대로다: **① 광선으로 실제 조명면을 찾고 ② 그 위에서 PO 표면적분**(`src/rcs_sbr.py:184`). ① 은 Sionna 가 이미 들고 있는 Mitsuba/OptiX 엔진을 그대로 부른다.

| 단계 | 무엇을 | 누가 |
|---|---|---|
| 첫 충돌 탐색 · 가림 | 어느 면이 실제로 조명되는가 | 🟢 Sionna 의 Mitsuba/OptiX 광선엔진 |
| 재질 \|Γ\| | 부품별 반사계수 | 🟢 Sionna 재질표 (`src/materials.py:53`) |
| PO 면적분 → σ | E = Σ \|Γᵢ\| e^{j2k pᵢ·û} d², σ = 4π\|E\|²/λ² | 🔵 우리 (`src/rcs_sbr.py:184`) |
| 셸 투과 | 얇은 유전체 셸 뒤 금속(배터리·PCB)의 코히런트 합 | 🔵 우리 (동 `penetrate=True`) |

**Sionna 는 광선을 쏘고 튀긴다** — 기술보고서(v1.2, 59쪽)에 SBR 이 48 ⟨outputs/prior_settled_sionna.json : word_counts_rerun_this_session.sionna_rt_technical_report_v2_59p.SBR or shooting-and-bouncing⟩회 나오고 우리도 그 엔진을 그대로 부른다. 같은 문서에서 `physical optics` 0 ⟨outputs/prior_settled_sionna.json : word_counts_rerun_this_session.sionna_rt_technical_report_v2_59p.physical optics⟩회 · `radar cross section` 0 ⟨outputs/prior_settled_sionna.json : word_counts_rerun_this_session.sionna_rt_technical_report_v2_59p.radar cross section⟩회 · `surface current` 0 ⟨outputs/prior_settled_sionna.json : word_counts_rerun_this_session.sionna_rt_technical_report_v2_59p.surface current⟩회이고, 거친 면은 정규화 산란패턴을 쓰는 경험 모델이다 — **표면전류 적분과 σ 출력을 우리가 얹는다**. 게재된 최신 선행(Clutter-Aware ISAC, Proc. IEEE 114(1))은 드론 메쉬를 Sionna 에 넣고 stock Fresnel 응답을 그대로 받는다(01편 §3).

ITU `metal` 의 산란계수 S = 0.0 ⟨outputs/report3_rt.json : C_metal.itu_metal_S⟩ 이라 stock 산란 모델이 금속에서 내놓는 항은 0 이고, 우리 σ 는 면적분에서 창발한다. 금속 4그룹(모터·배터리·PCB·카메라)만 남긴 메쉬의 방위평균 σ 는 전체의 112% ⟨outputs/report3_rt.json : C_metal.metal_share_pct⟩ 다 — 코히런트 합이라 100 % 를 넘는다.

![lit versus shadowed](outputs/figures/mesh_compare_material_shadow.png)

**그림 4.** PO 적분이 실제로 올라타는 면은 어디까지인가?

조명원을 방위 280° ⟨outputs/report02_derived.json : occlusion.az_deg⟩, 고각 15° ⟨outputs/report02_derived.json : occlusion.el_deg⟩ 에 두었다 — 방위 72 ⟨outputs/report02_derived.json : occlusion.n_az_sweep⟩점 스윕에서 7기체 평균 그늘비율의 중앙값에 가장 가까운 방위다(규칙이 고른다). 그 자세에서 조명원을 향한 외피의 29 ⟨outputs/report02_derived.json : occlusion.shadow_min_pct⟩~47% ⟨outputs/report02_derived.json : occlusion.shadow_max_pct⟩ 가 기체 자신에 가려 있다.

가림 판정은 생산 SBR 이 쓰는 그림자광선 그대로다(`rcs_sbr._exit_visible()`), 그림의 색은 재질이 아니라 조명 상태를 부호화한다.

| 기체 | 외피 그늘 | 가림 [dB] | 셸 투과 [dB] | 합 [dB] | 이산화 바닥 [dB] | 생산 σ [dBsm] |
|---|---|---|---|---|---|---|
| Mini 5 Pro ⭐ | 41 % | +5.98 | +3.66 | +2.32 | 0.014 | -22.0 |
| Mavic 4 Pro | 29 % | +3.75 | +2.55 | +1.20 | 0.021 | -18.2 |
| Matrice 4E ⭐ | 35 % | +6.63 | +3.72 | +2.90 | 0.021 | -18.9 |
| Phantom 4 | 36 % | +5.90 | +4.06 | +1.84 | 0.056 | -19.9 |
| X500 V2 | 47 % | +1.07 | +0.00 | +1.07 | 0.037 | -16.8 |
| Typhoon H (H480) | 39 % | +2.91 | +1.51 | +1.40 | 0.014 | -15.8 |
| S1000+ | 42 % | +0.11 | -0.19 | +0.30 | 0.071 | -12.3 |

출처 ⟨outputs/report02_derived.json : occlusion.rows⟩

가림을 끄면 방위평균 σ 가 6.63 dB ⟨outputs/report02_derived.json : occlusion.max_db⟩(Matrice 4E ⭐ ⟨outputs/report02_derived.json : occlusion.max_drone⟩, 닫힌 동체)까지 부풀고, 열린 프레임인 S1000+ ⟨outputs/report02_derived.json : occlusion.min_drone⟩ 에서는 0.11 dB ⟨outputs/report02_derived.json : occlusion.min_db⟩ 다. 7 ⟨outputs/report02_derived.json : occlusion.n_above_floor⟩기체 전부에서 이 값이 이산화 바닥(최대 0.071 dB ⟨outputs/report02_derived.json : occlusion.floor_max_db⟩) 위에 있다.

### §2.1 바이스태틱 — 수신 방향으로도 그림자 광선을 쏜다

각 충돌점에서 수신기 방향으로 그림자 광선을 한 번 더 쏘아 **출사 쪽 가림**을 판정한다(`src/rcs_sbr.py:330` `rcs_sbr_multistatic`). Sagitta(preprint, arXiv:2604.09243 각주 1)가 바이스태틱 SBR 에서 빠져 있다고 지목한 바로 그 단계다.

켠 효과는 상반성으로 잰다 — 위반 최대치가 9.69 ⟨outputs/sbr_defect_fixes.json : d2_exit_vis_effect_on_reciprocity.worst_without_exit_vis_db⟩ → 8.24 dB ⟨outputs/sbr_defect_fixes.json : d2_exit_vis_effect_on_reciprocity.worst_with_exit_vis_db⟩ 로 내려간다. 모노스태틱에서 이 검사는 무연산이라 생산 σ 는 0.000e+00 dB ⟨outputs/sbr_defect_fixes.json : d4_epsilon_sensitivity.combos[0].by_drone.mavic4pro.monostatic_noop_max_abs_db⟩ 그대로다.

**바이스태틱 자세 패턴은 β ≤ 45° 에서 성립한다** — 그 범위의 상반성 RMS 가 2.57 dB ⟨outputs/sbr_defect_fixes.json : d2_exit_vis_effect_on_reciprocity.rms_with_exit_vis_db[3]⟩ (β=45° ⟨outputs/sbr_defect_fixes.json : d2_exit_vis_effect_on_reciprocity.beta_deg[3]⟩) 다.

## §3. 기준해 셋과 대조했다

구 후방산란은 두 개의 **닫힌형 기준해**를 갖는다 — 정확 Mie 와 해석 PO(`benchmark/mie_pec_sphere.py:98`, `:127`). 둘 다 우리 출력이 아니라 과녁이다.

```
(커널 − Mie)  =  (커널 − 해석 PO)   +   (해석 PO − Mie)
                  ↑ 우리 수치오차          ↑ PO 모델 자체의 간극
```
커널이 PO 이므로 **수치 수렴의 과녁은 해석 PO** 이고, Mie 잔차는 PO 모델 자체의 간극이라는 두 번째 눈금이다. 둘을 나눠 두면 각각이 얼마인지 그대로 읽힌다.

<!--pk:figure {"kind": "figure", "path": "outputs/figures/report02_f5_reference_gap.png", "figure_no": "5", "question": "일곱 기체가 놓인 kr 자리에서 우리 수치오차와 PO 모델의 간극은 각각 얼마인가?", "paper_caption": "Our physical-optics kernel against two closed-form reference solutions on a PEC sphere, the two errors they measure separated, and the electrical size of the seven airframes on the same axis.", "vector_pdf": "outputs/figures/report02_f5_reference_gap.pdf", "report": "report02_target"}-->
![report02_f5_reference_gap.png](outputs/figures/report02_f5_reference_gap.png)

**그림 5.** 일곱 기체가 놓인 kr 자리에서 우리 수치오차와 PO 모델의 간극은 각각 얼마인가?

### §3.1 두 눈금

|  | 우리 수치오차 · 기준해 = 해석 PO | PO 모델의 간극 · 기준해 = 정확 Mie |
|---|---|---|
| 최대 편차 (kr=1..100) | 0.201 dB ⟨outputs/sbr_kr_sweep.json : summary_div16.max_abs_db_vs_po⟩ | 6.73 dB ⟨outputs/sbr_kr_sweep.json : summary_div16.max_abs_db_vs_mie⟩ (kr=1) |
| kr≥30 산포 | 0.885% ⟨outputs/sbr_kr_sweep.json : summary_div16.std_sbr_over_po_pct_kr_ge30⟩ | 1.834% ⟨outputs/sbr_kr_sweep.json : summary_div16.std_sbr_over_mie_pct_kr_ge30⟩ |
| 1 dB 안으로 드는 kr | 전 구간 (kr=1 ⟨outputs/sbr_kr_sweep.json : summary_div16.kr_min⟩ 부터) | kr ≥ 9.06 ⟨outputs/report02_derived.json : po_floor.kr_below_1p0_db⟩ |
| 0.5 / 0.2 dB 안으로 | 전 구간 (0.201 dB ⟨outputs/sbr_kr_sweep.json : summary_div16.max_abs_db_vs_po⟩ 이내) | kr ≥ 15.16 ⟨outputs/report02_derived.json : po_floor.kr_below_0p5_db⟩ / 30.87 ⟨outputs/report02_derived.json : po_floor.kr_below_0p2_db⟩ |

기체 7 × 밴드 3 = 21 ⟨outputs/report02_derived.json : electrical.n_airframe_band⟩ 조합 중 1 ⟨outputs/report02_derived.json : electrical.n_below_po_1db⟩개가 오른쪽 열의 1 dB 문턱 아래에 놓이고, 그것이 실측 대상 DJI Mini 5 Pro ⟨outputs/report02_derived.json : electrical.kr_min_name⟩ 다 — §4 의 앵커가 정확히 그 자리를 잡는다.

### §3.2 다중반사 위상 — PEC 이면각 닫힌형과 대조

직각 이면각의 이등분선 입사는 σ = 8πa²b²/λ² 로 닫혀 있다. 2회 반사를 켜고 변 길이 4점에서 그 값과 맞댄다(`benchmark/verify_sbr_defect_fixes.py`, 3.5 GHz ⟨outputs/report02_derived.json : bands_ghz.5G⟩, λ/12 격자).

| 변 a [m] | 해석해 [dBsm] | 1회 반사 [dBsm] | 2회 반사 [dBsm] | 오차 [dB] |
|---|---|---|---|---|
| 0.15 | 2.39 | -14.63 | 2.95 | +0.556 |
| 0.20 | 7.39 | -13.79 | 7.85 | +0.466 |
| 0.30 | 14.43 | -118.33 | 14.51 | +0.076 |
| 0.40 | 19.43 | -7.76 | 19.32 | -0.108 |

출처 ⟨outputs/sbr_defect_fixes.json : d3_multibounce_phase.rows⟩

오목부에서 오는 항이 어디에 있는지 1회/2회 열이 바로 보여준다. 같은 스크립트가 매끄러운 기준체도 함께 잰다 — 구는 λ/16 격자에서 해석 PO 대비 -0.021 dB ⟨outputs/sbr_defect_fixes.json : d3_multibounce_phase.sphere_and_plate.sphere_vs_po_db.sphere_lam/16_vs_po⟩, 평판은 λ/10 격자에서 -0.011 dB ⟨outputs/sbr_defect_fixes.json : d3_multibounce_phase.sphere_and_plate.plate_db.plate_lam/10⟩ 다.

## §4. 앵커 — 주파수 축만 측정에서 받았다

게재된 표준트랙 논문의 분해를 그대로 쓴다(Zhang, IEEE JSAC 44:702, 2026 — 측정 적합 모델): σ = A(f)·B₁(φ,θ)·B₂. **A(f) 의 기울기는 측정에서, A(f) 의 레벨과 B₁ 은 우리 계산에서 온다.**

PO 면적분은 f² 로 커지는 정반사항을 담는다. 기하에서 나온 우리 밴드 기울기는 0.742 ⟨outputs/report02_derived.json : band_slope.ours_min⟩(x500v2 ⟨outputs/report02_derived.json : band_slope.ours_min_drone⟩) ~ 1.699 dB/GHz ⟨outputs/report02_derived.json : band_slope.ours_max⟩(phantom4 ⟨outputs/report02_derived.json : band_slope.ours_max_drone⟩) 이고, 측정은 0.210 ⟨outputs/rcs_anchor.json : literature.mu_eps.multiband_phantom3.mu_a⟩(Das, IEEE WCL 2026) 와 0.315 dB/GHz ⟨outputs/rcs_anchor.json : literature.mu_eps.mono3d_theta90.mu_a⟩(Yuan, EuCAP 2025) 다. 모서리 회절항(PTD)은 면적분과 별개의 항이므로 주파수축은 측정에서 받고, 그 항을 넣는 일은 §6 의 세 번째 줄이다.

<!--pk:figure {"kind": "figure", "path": "outputs/figures/report02_f6_band_slope.png", "figure_no": "6", "question": "측정 앵커는 각 기체의 밴드 기울기를 어디로 옮기는가?", "paper_caption": "Band slope of the azimuth-mean RCS computed from geometry for each airframe against the two measured slopes, and where slope_only re-anchoring places every airframe.", "vector_pdf": "outputs/figures/report02_f6_band_slope.pdf", "report": "report02_target"}-->
![report02_f6_band_slope.png](outputs/figures/report02_f6_band_slope.png)

**그림 6.** 측정 앵커는 각 기체의 밴드 기울기를 어디로 옮기는가?

### §4.1 모드 선택 — 무엇을 옮기고 무엇을 대가로 내는가

`src/sigma_anchor.py` 는 재보정 모드 셋을 제공한다. 생산 기준은 `slope_only ⟨outputs/report02_derived.json : anchor_modes.production_mode⟩` 이고, 밴드별로 옮기는 양은 +2.70 ⟨outputs/report02_derived.json : anchor.correction_max_db⟩(phantom4 ⟨outputs/report02_derived.json : anchor.correction_max_drone⟩ @ LTE 1.843 GHz ⟨outputs/report02_derived.json : anchor.correction_max_band⟩) ~ -2.41 dB ⟨outputs/report02_derived.json : anchor.correction_min_db⟩(s1000plus ⟨outputs/report02_derived.json : anchor.correction_min_drone⟩ @ WiFi 5.21 GHz ⟨outputs/report02_derived.json : anchor.correction_min_band⟩) 다.

| 모드 | 무엇을 옮기나 | 평균 레벨이동 [dB] | 대가 |
|---|---|---|---|
| slope_only (기본) | 주파수 의존성 A(f) 의 기울기만 | 0.00 | 없음 — 크기 가정 0개 |
| level_and_slope_L2 | 레벨 + 기울기 | -0.82 ~ +7.42 | 크기전이 L² 가정 (σ ∝ 투영면적) |
| level_and_slope_L4 | 레벨 + 기울기 | +1.93 ~ +16.92 | 크기전이 L⁴ 가정 (σ ∝ A²/λ²) · L² 와 최대 9.50 dB 차 (DJI S1000+) |

출처 ⟨outputs/report02_derived.json : anchor_modes.rows⟩

레벨까지 앵커에 맞추려면 크기전이 법칙을 하나 골라야 하고, 그 선택이 기체에 따라 최대 9.50 dB 를 정한다. 측정이 그 대가 없이 제약하는 양은 기울기뿐이므로 기울기만 받는다. ⟨outputs/report02_derived.json : anchor_modes.why_production⟩

레벨이동 열은 기체 7 ⟨outputs/report02_derived.json : anchor_modes.n_airframes⟩종의 세 밴드 평균 Δ 다 — 정의는 `anchor_modes.definition` 에 있다.

### §4.2 세 인자, 각각의 출처

| 인자 | 무엇 | 어디서 | 이 편의 근거 |
|---|---|---|---|
| A(f) 기울기 | 주파수 의존성 | **측정**(Das) | μ 기울기 0.21 dB/GHz ⟨outputs/rcs_anchor.json : literature.mu_eps.multiband_phantom3.mu_a⟩ |
| A(f) 레벨 | 절대 레벨 | **우리 PO 출력** | `slope_only ⟨outputs/report02_derived.json : anchor_modes.production_mode⟩` 의 평균 레벨이동 0.00 dB ⟨outputs/report02_derived.json : anchor_modes.level_shift_abs_max_db⟩ |
| B₁(φ,θ) | 자세에 따른 모양 | **기하**(광선 가림 + PO) | 재보정 후 정규화 패턴 변화 1.9e-15 dB ⟨outputs/report02_derived.json : anchor.shape_invariance_max_abs_db⟩ |
| B₂ | 자세 요동의 분포족 | 기하 | 문헌 적합 RMSE 2.41 dB ⟨outputs/rcs_anchor.json : literature.fit_rmse_db.AAV⟩ 가 기준선 |

정렬 후 7기종 기울기가 모두 0.210 dB/GHz ⟨outputs/report02_derived.json : anchor.slope_after_db_per_ghz⟩ 위에 서고, 기종 간 산포는 1.9e-15 dB/GHz ⟨outputs/report02_derived.json : anchor.slope_after_spread_db_per_ghz⟩ 다. 주파수 눈금만 측정에서 오고 레벨과 모양은 그대로다.

### §4.3 비교가능성 원장 — 기체마다 앵커와의 거리가 다르다

앵커 기체는 DJI Phantom 3 ⟨outputs/report02_derived.json : anchor.anchor_platform⟩ 한 대다. 같은 급이면 `direct`(1 ⟨outputs/report02_derived.json : anchor.n_direct⟩대), 크기법칙으로 옮기면 `scaled`(5 ⟨outputs/report02_derived.json : anchor.n_scaled⟩대), 위상 자체가 다르면 `not_comparable`(1 ⟨outputs/report02_derived.json : anchor.n_not_comparable⟩대)로 적는다.

| 기체 | 대각 D [m] | D/D_ref | 로터 | 판정 | L²↔L⁴ 산포 [dB] |
|---|---|---|---|---|---|
| DJI Mini 5 Pro | 0.275 | 0.79 | 4 | scaled | 2.09 |
| DJI Phantom 4 | 0.350 | 1.00 | 4 | direct | 0.00 |
| DJI Mavic 4 Pro | 0.441 | 1.26 | 4 | scaled | 2.01 |
| DJI Matrice 4E | 0.439 | 1.25 | 4 | scaled | 1.96 |
| Yuneec Typhoon H (H480) | 0.480 | 1.37 | 6 | scaled | 2.74 |
| Holybro X500 V2 | 0.500 | 1.43 | 4 | scaled | 3.10 |
| DJI S1000+ | 1.045 | 2.99 | 8 | not_comparable | 9.50 |

출처 ⟨outputs/sigma_anchor.json : drones⟩

### §4.4 앵커가 통제한 항목과 남은 항목의 크기

각 항목을 상태와 dB 크기로 함께 싣는다. 05편의 밴드 간 비교는 이 표와 나란히 읽는다.

| 항목 | 상태 | 크기 [dB] |
|---|---|---|
| polarisation | UNRESOLVED | 미상 |
| statistic convention (Das mu) | RESOLVED_EMPIRICALLY | +0.93 |
| size transfer law | UNRESOLVED | +9.50 |
| single platform / single lab | UNRESOLVED | 미상 |
| elevation matching | PARTIAL | +0.73 |
| near-field vs far-field, environment | OK | +0.00 |

출처 ⟨outputs/sigma_anchor.json : uncontrolled⟩

가장 큰 항은 **size transfer law ⟨outputs/report02_derived.json : anchor.largest_uncontrolled_term⟩** 9.50 dB ⟨outputs/report02_derived.json : anchor.largest_uncontrolled_db⟩ 이고, 실제 적용한 보정 최대치 2.70 dB ⟨outputs/report02_derived.json : anchor.correction_abs_max_db⟩ 보다 크다. 그래서 커널은 그대로 두고 **원장으로만** 적용한다 — 05편이 이 표를 함께 읽는다.

## §5. σ 오차를 넣어도 파형 순위가 서는 범위

σ 오차를 두 갈래로 나눠 검출거리 R90 순위에 넣었다(`benchmark/sigma_sensitivity.py`, 기체 5 ⟨outputs/report02_derived.json : sigma_sens.n_airframes⟩ × 밴드 3 ⟨outputs/report02_derived.json : sigma_sens.n_bands⟩ = 15 ⟨outputs/report02_derived.json : sigma_sens.n_cells⟩셀). **공통모드**는 한 기체의 세 밴드를 같은 dB 로 옮기고(절대 레벨 오차의 모양), **차분**은 밴드마다 다르게 옮긴다(기울기 오차의 모양).

공통모드에서 순위는 -10 ⟨outputs/report02_derived.json : sigma_sens.offset_min_db⟩~+10 dB ⟨outputs/report02_derived.json : sigma_sens.offset_max_db⟩ 전 구간 15 ⟨outputs/report02_derived.json : sigma_sens.n_cells⟩셀에서 그대로다 — 움직이는 것은 절대거리뿐이고 그 기울기가 σ 1 dB 당 0.246 dB ⟨outputs/report02_derived.json : sigma_sens.slope_mean_db_per_db⟩(0.223 ⟨outputs/report02_derived.json : sigma_sens.slope_min_db_per_db⟩~0.256 ⟨outputs/report02_derived.json : sigma_sens.slope_max_db_per_db⟩), 즉 σ ±10 dB 에서 거리 -43 ⟨outputs/report02_derived.json : sigma_sens.range_at_minus10_pct⟩ %~+76% ⟨outputs/report02_derived.json : sigma_sens.range_at_plus10_pct⟩ 다. **논문이 절대 σ 에 기대는 곳은 여기서 끝난다.**

차분오차가 순위를 정하는 축이고, 그래서 §4 의 앵커가 잡는 축이 정확히 이것이다. 자세평균 σ 로 인용하면 다섯 기체가 한 순위(LTE > 5G > WiFi ⟨outputs/report02_derived.json : sigma_sens.aspect_avg_order⟩)에 합의하고(단일자세에서는 순위 3 ⟨outputs/report02_derived.json : sigma_sens.single_aspect_n_orders⟩종), 거기에 측정 기울기를 얹으면 최악 뒤집힘 문턱이 0.09 ⟨outputs/report02_derived.json : sigma_sens.worst_flip_aspect_avg_db⟩ → 3.72 dB ⟨outputs/report02_derived.json : sigma_sens.worst_flip_anchored_db⟩ 로 올라간다(+3.63 dB ⟨outputs/report02_derived.json : sigma_sens.anchor_gain_db⟩).

<!--pk:figure {"kind": "figure", "path": "outputs/figures/report02_f7_sigma_sensitivity.png", "figure_no": "7", "question": "σ 오차가 어느 축으로 얼마나 커질 때 세 파형의 순위가 뒤집히는가?", "paper_caption": "A common-mode RCS error leaves the three-waveform ranking unchanged over the full sweep and moves only absolute range, while a per-band differential error is the axis that decides the ranking and is the axis the measured frequency slope anchors.", "vector_pdf": "outputs/figures/report02_f7_sigma_sensitivity.pdf", "report": "report02_target"}-->
![report02_f7_sigma_sensitivity.png](outputs/figures/report02_f7_sigma_sensitivity.png)

**그림 7.** σ 오차가 어느 축으로 얼마나 커질 때 세 파형의 순위가 뒤집히는가?

| 기체 | 최대 치수 [m] | D/λ @LTE | 뒤집힘 문턱 · 단일자세 [dB] | 뒤집힘 문턱 · 자세평균 [dB] | 밴드간 σ 산포 [dB] | P(순위 보존) @1 dB |
|---|---|---|---|---|---|---|
| DJI Mini 5 Pro ⭐ | 0.378 | 2.32 | 7.42 | 2.95 | 2.2 | 0.992 |
| DJI Phantom 4 | 0.471 | 2.89 | 5.03 | 2.20 | 14.8 | 0.947 |
| DJI Mavic 4 Pro | 0.556 | 3.42 | 1.84 | 0.09 | 18.7 | 0.745 |
| DJI Matrice 4E ⭐ | 0.587 | 3.61 | 0.61 | 1.30 | 5.5 | 0.583 |
| DJI S1000+ | 1.348 | 8.29 | 0.79 | 0.80 | 10.4 | 0.419 |

출처 ⟨outputs/report02_derived.json : sigma_sens.rows⟩

### §5.1 이 표를 논문이 쓰는 법

뒤집힘을 정하는 것은 기체 크기가 아니라 **밴드간 σ 산포**다 — 가장 작은 DJI Mini 5 Pro ⭐ ⟨outputs/report02_derived.json : sigma_sens.flip_single_max_name⟩ 가 문턱 7.42 dB ⟨outputs/report02_derived.json : sigma_sens.flip_single_max_db⟩ 로 가장 견고하고, DJI Matrice 4E ⭐ ⟨outputs/report02_derived.json : sigma_sens.flip_single_min_name⟩ 가 0.61 dB ⟨outputs/report02_derived.json : sigma_sens.flip_single_min_db⟩ 로 가장 예민하다.

앵커 전 차분 폭은 5.01 dB ⟨outputs/report02_derived.json : sigma_sens.realistic_span_db⟩(생산 기울기 − 측정 기울기 × 밴드 스팬)이고, 그 폭 안에서 순위가 바뀌는 기체가 3 ⟨outputs/report02_derived.json : sigma_sens.n_flip_inside_realistic⟩대다. 그래서 인용 규약을 둘로 고정한다 — **자세평균 σ 로 인용하고, `modes.slope_only.delta_db` 를 적용한 뒤 읽는다**.

R90 사슬(`src/experiment_freespace_sigma.py`)은 σ 격자를 앵커 전 값으로 읽는다. 지금 델타를 적용하면 5 ⟨outputs/report02_derived.json : sigma_sens.n_airframes⟩기체 중 2 ⟨outputs/report02_derived.json : sigma_sens.anchor_not_applied_n_order_changed⟩대의 순위가 바뀐다 — 적용 지점은 §6 표 첫 줄이고, 05편이 그 위에서 밴드를 읽는다.

σ 격자가 바뀌면 결과도 바뀐다는 것을 크기로 적는다 — 07-24 격자와 07-29 블레이드 갱신본 사이에서 R90 이 최대 17.9% ⟨outputs/report02_derived.json : sigma_sens.blade_update_max_range_pct⟩ 움직이고 순위쌍 1 ⟨outputs/report02_derived.json : sigma_sens.blade_update_n_orders_changed⟩개가 뒤집힌다. 자세평균 자체는 방위 격자에 둔감하다 — 120점과 720점 격자의 방위평균 차이가 0.075 dB ⟨outputs/sigma_anchor.json : drones.mini5pro.grid_vs_720az_db.LTE 1.843 GHz⟩ (Mini 5 Pro @ LTE) 다.

In [ ]:
# 이 편의 숫자를 직접 열어보기 — 그림·표의 모든 값은 아래 JSON 에서 나온다.
import json
D = json.load(open('outputs/report02_derived.json'))
P = D['provenance']
print('메쉬 소스 최신 편집 :', P['mesh_source_newest'],
      f"— 읽는 JSON {P['n_sources']}개 중 새것 {P['n_fresh']} · 옛것 {P['n_stale']}")
for r in P['rows']:
    print(f"  {'새것' if r['fresh'] else '옛것'}  {r['source']:28s} "
          f"{r['stamp']}  {r['used_in']}")
print('사진 대조 :', {r['airframe']: round(r['iou'], 3) for r in D['photo']['rows']})
print('가림 [dB] :', {r['airframe']: round(r['d_occlusion_db'], 2)
                     for r in D['occlusion']['rows']})
print('앵커 원장 :', {k: D['anchor'][k] for k in
      ('mode', 'from_measurement', 'from_ours', 'largest_uncontrolled_db')})

## §6. 다음 단계

| 다음에 할 일 | 그러면 결정되는 것 | 어디서 |
|---|---|---|
| R90 사슬에 `modes.slope_only.delta_db` 를 적용한다 | 앵커 후 파형 순위가 확정된다 — 지금 적용하면 2 ⟨outputs/report02_derived.json : sigma_sens.anchor_not_applied_n_order_changed⟩기체의 순위가 바뀐다 | `src/experiment_freespace_sigma.py` → 05편 §5 |
| 자세 패턴과 부품별 스트립을 현재 메쉬로 다시 재어 이 편에 되싣는다 | 07-31 기하 위의 자세 패턴 σ 가 확정된다 | `src/viz_report2.py` → `outputs/report2_waveform_rcs.json` |
| §4 의 앵커 사슬(rcs_anchor → sigma_anchor)을 현재 메쉬로 다시 돌린다 | 밴드 기울기와 재보정 원장이 현재 기하 위에 선다 | `benchmark/rcs_anchor.py` → `src/sigma_anchor.py` |
| §1 의 그림 넷을 게재 규격(벡터 + 300 dpi · 8 pt)으로 다시 그린다 | 원고 그림 전부가 2단 조판 축소에서 살아남는다 | `src/viz_mesh_gallery.py` · `src/viz_mesh_material.py` → `paper_kit.save_figure` |
| PO 면적분에 등가 모서리 전류(PTD)를 더한다 | 밴드 기울기를 기하만으로 세울 수 있는지 결정된다 | `src/rcs_sbr.py:184` → 02편 §4 기울기 재측정 |
| Matrice 4E · Mini 5 Pro 의 짐벌·착륙장치를 사진 실루엣에 맞춘다 | 실측 두 기체의 IoU 가 상한 대비 어디까지 오르는지 결정된다 | `src/drone_cad.py` → §1.1 재측정 |
| Matrice 4E · Mini 5 Pro 의 상대 레벨을 실측한다 | 크기전이 지수가 직접 고정되어 §4.4 의 최대 항이 닫힌다 | 06편 §2 측정 설계 |
| 교정구를 표적과 같은 자리에서 함께 잰다 | 지금 우리 PO 출력인 절대 레벨이 처음으로 측정에 앵커된다 | 06편 §2-2 → `src/sigma_anchor.py` 레벨 앵커 승격 |
| VV/HH 2편파를 잰다 | §4.4 편파 항의 크기가 수치로 확정된다 | 06편 §2 → `src/materials.py:171` 편파 분해 결정 |
| 평판·이면각 표준체로 같은 kr 스윕을 돌린다 | 얇고 모서리 많은 표적에서의 PO 간극 문턱이 선다 | `benchmark/verify_sbr_defect_fixes.py` 의 두 닫힌형 재사용 |
| 2회 반사를 β 별로 다시 돌린다 | 바이스태틱 유효범위가 45° 위로 얼마나 넓어지는지 결정된다 | `src/rcs_sbr.py:330` 상반성 검사 |
| 회전 블레이드 마이크로도플러를 검증 가능한 형태로 세운다 | 미세도플러 서명을 이 커널 위에서 인용할 수 있게 된다 | future work (Costa, IEEE JSTEAP 의 해석 경로가 기준선) |

<!--pk:methods {"kind": "methods", "report": "report02_target", "text": "Each airframe is a watertight triangle mesh built from the published outer dimensions, the motor-to-motor diagonal and the propeller diameter, with every face keeping its part-level material group (metal, PCB, camera assembly, carbon, plastic shell, propeller); the reflection coefficients are the ITU-R P.2040 values that Sionna 2.0.1 itself uses. For one incidence direction we call Sionna's Mitsuba/OptiX ray engine for a first-hit visibility test on a ray grid of pitch lambda/12, then integrate the physical-optics surface current over the lit facets only, E = sum_i |Gamma_i| exp(j 2 k p_i . u) dA and sigma = 4 pi |E|^2 / lambda^2; thin dielectric shells are transmitted with the round-trip factor tau = 1 - |Gamma|^2 and the metal behind them is summed coherently, and for a bistatic pair a second shadow ray is cast from every hit point toward the receiver. The kernel is checked against three closed-form reference solutions - the analytic physical-optics sphere, the exact Mie PEC sphere and the PEC dihedral 8 pi a^2 b^2 / lambda^2 - over kr = 1 to 100 at 21 points and 48 incidence directions. The absolute level of sigma is the kernel output; only the frequency slope is re-anchored, onto the measured 0.210 dB/GHz of Das et al., which rotates sigma(f) about the band-mean level and leaves the normalised aspect pattern unchanged. Software: Sionna 2.0.1, Sionna-RT 2.0.1, Mitsuba 3.8.0, Dr.Jit 1.3.1, NumPy 2.5.0, Python 3.12.", "tools": ["Sionna 2.0.1", "Mitsuba 3.8.0", "Python 3.12"], "params": ["0.210 dB", "21 points", "E = sum_i", "kr = 1", "sigma = 4", "tau = 1"], "versions": ["Jit 1.3.1", "Mitsuba 3.8.0", "NumPy 2.5.0", "Python 3.12", "Sionna 2.0.1", "Sionna-RT 2.0.1"], "n_words": 235}-->
### §7. 방법 문단 (논문 이관용)

Each airframe is a watertight triangle mesh built from the published outer dimensions, the motor-to-motor diagonal and the propeller diameter, with every face keeping its part-level material group (metal, PCB, camera assembly, carbon, plastic shell, propeller); the reflection coefficients are the ITU-R P.2040 values that Sionna 2.0.1 itself uses. For one incidence direction we call Sionna's Mitsuba/OptiX ray engine for a first-hit visibility test on a ray grid of pitch lambda/12, then integrate the physical-optics surface current over the lit facets only, E = sum_i |Gamma_i| exp(j 2 k p_i . u) dA and sigma = 4 pi |E|^2 / lambda^2; thin dielectric shells are transmitted with the round-trip factor tau = 1 - |Gamma|^2 and the metal behind them is summed coherently, and for a bistatic pair a second shadow ray is cast from every hit point toward the receiver. The kernel is checked against three closed-form reference solutions - the analytic physical-optics sphere, the exact Mie PEC sphere and the PEC dihedral 8 pi a^2 b^2 / lambda^2 - over kr = 1 to 100 at 21 points and 48 incidence directions. The absolute level of sigma is the kernel output; only the frequency slope is re-anchored, onto the measured 0.210 dB/GHz of Das et al., which rotates sigma(f) about the band-mean level and leaves the normalised aspect pattern unchanged. Software: Sionna 2.0.1, Sionna-RT 2.0.1, Mitsuba 3.8.0, Dr.Jit 1.3.1, NumPy 2.5.0, Python 3.12.

버전 — `Sionna 2.0.1` · `Mitsuba 3.8.0` · `Python 3.12`

<!--rs:paper-->
<!--pk:defence {"kind": "defence", "report": "report02_target", "rows": [{"주장": "표적 σ 는 부품별 재질 메쉬 위의 PO 면적분으로 계산하고, 커널은 해석 PO 기준해와 kr 1~100 전 구간에서 0.201 dB ⟨outputs/sbr_kr_sweep.json : summary_div16.max_abs_db_vs_po⟩ 안에서 맞는다", "근거": "그림 5 · 표 §3.1 · `outputs/sbr_kr_sweep.json:summary_div16.max_abs_db_vs_po`", "공격": "PO 는 few-λ 표적에서 부정확하다 — 드론이 바로 그 크기다", "답": "PO 모델 자체의 간극을 정확 Mie 기준해로 따로 쟀다 — kr ≥ 9.06 ⟨outputs/report02_derived.json : po_floor.kr_below_1p0_db⟩ 에서 1 dB, kr ≥ 15.16 ⟨outputs/report02_derived.json : po_floor.kr_below_0p5_db⟩ 에서 0.5 dB 안이다 ⟨outputs/report02_derived.json : po_floor⟩. 21 ⟨outputs/report02_derived.json : electrical.n_airframe_band⟩ 조합 중 1 ⟨outputs/report02_derived.json : electrical.n_below_po_1db⟩ 개가 그 문턱 아래이고, 그 자리를 §4 앵커가 잡는다"}, {"주장": "가림 판정은 Sionna 의 Mitsuba/OptiX 광선엔진이 하고, 표면전류 적분과 σ 출력을 우리가 얹었다", "근거": "§2 표 · `outputs/prior_settled_sionna.json:word_counts_rerun_this_session`", "공격": "Sionna 에도 SBR 이 있으니 엔진 기여는 이미 그 안에 있다", "답": "기술보고서(v1.2, 59쪽)에 SBR 은 48 ⟨outputs/prior_settled_sionna.json : word_counts_rerun_this_session.sionna_rt_technical_report_v2_59p.SBR or shooting-and-bouncing⟩회 나오고 우리도 그 엔진을 그대로 쓴다. 같은 문서에서 `physical optics` 0 ⟨outputs/prior_settled_sionna.json : word_counts_rerun_this_session.sionna_rt_technical_report_v2_59p.physical optics⟩회 · `radar cross section` 0 ⟨outputs/prior_settled_sionna.json : word_counts_rerun_this_session.sionna_rt_technical_report_v2_59p.radar cross section⟩회이므로, 더한 것은 표면적분과 σ 출력이다"}, {"주장": "절대 레벨은 우리 PO 출력이고, 측정에서 받은 것은 주파수 기울기 하나다", "근거": "그림 6 · 표 §4.2 · `outputs/report02_derived.json:anchor_modes`", "공격": "절대 σ 가 측정으로 검증되지 않았다면 검출 결과 전체가 흔들린다", "답": "공통모드 σ 오차는 세 밴드를 함께 옮겨 15 ⟨outputs/report02_derived.json : sigma_sens.n_cells⟩셀 전부에서 순위를 그대로 두고, 절대거리만 σ 1 dB 당 0.246 dB ⟨outputs/report02_derived.json : sigma_sens.slope_mean_db_per_db⟩ 움직인다 ⟨outputs/sigma_sensitivity.json : common_mode⟩. 논문은 순위를 주장하고, 절대 레벨은 06편 교정구로 앵커한다"}, {"주장": "밴드별 차분 σ 오차의 뒤집힘 문턱은 기체별 0.61 ⟨outputs/report02_derived.json : sigma_sens.flip_single_min_db⟩~7.42 dB ⟨outputs/report02_derived.json : sigma_sens.flip_single_max_db⟩ 다", "근거": "그림 7 · 표 §5 · `outputs/sigma_sensitivity.json:differential`", "공격": "그 문턱이 현실 차분 폭 5.01 dB ⟨outputs/report02_derived.json : sigma_sens.realistic_span_db⟩ 안에 드는 기체가 셋이다 — 그 폭 안에서 순위가 바뀐다", "답": "인용 규약을 두 개로 고정해서 답한다 — 자세평균 σ 로 인용하면 다섯 기체가 한 순위에 합의하고, 측정 기울기를 얹으면 최악 문턱이 0.09 ⟨outputs/report02_derived.json : sigma_sens.worst_flip_aspect_avg_db⟩→3.72 dB ⟨outputs/report02_derived.json : sigma_sens.worst_flip_anchored_db⟩ 로 올라간다. 남은 차분은 06편이 두 기체의 밴드별 상대 레벨로 닫는다"}, {"주장": "바이스태틱 자세 패턴은 β ≤ 45° 에서 성립하고, 그 범위의 상반성 RMS 는 2.57 dB ⟨outputs/sbr_defect_fixes.json : d2_exit_vis_effect_on_reciprocity.rms_with_exit_vis_db[3]⟩ 다", "근거": "§2.1 · `outputs/sbr_defect_fixes.json:d2_exit_vis_effect_on_reciprocity`", "공격": "β > 45° 에서 상반성이 크게 깨진다면 엔진 자체를 믿기 어렵다", "답": "출사 쪽 가림을 켜면 위반 최대치가 9.69 ⟨outputs/sbr_defect_fixes.json : d2_exit_vis_effect_on_reciprocity.worst_without_exit_vis_db⟩ → 8.24 dB ⟨outputs/sbr_defect_fixes.json : d2_exit_vis_effect_on_reciprocity.worst_with_exit_vis_db⟩ 로 내려간다. 논문은 β ≤ 45° 만 쓰고, 그 위는 2회 반사를 β 별로 다시 돌려 §6 에서 넓힌다"}, {"주장": "가림은 방위평균 σ 를 기체별 0.11 ⟨outputs/report02_derived.json : occlusion.min_db⟩~6.63 dB ⟨outputs/report02_derived.json : occlusion.max_db⟩ 옮긴다", "근거": "그림 4 · 표 §2 · `outputs/report02_derived.json:occlusion.rows`", "공격": "그 차이가 이산화 잡음일 수 있다", "답": "PO 를 λ/7↔λ/12 로 돌린 이산화 바닥이 최대 0.071 dB ⟨outputs/report02_derived.json : occlusion.floor_max_db⟩ 이고, 7 ⟨outputs/report02_derived.json : occlusion.n_above_floor⟩기체 전부에서 가림이 그 위에 있다"}, {"주장": "메쉬 형상은 사진 실루엣 · 제조사 CAD 치수 · 실물 유래 메쉬 셋으로 검사했다", "근거": "그림 2 · 표 §1.2 · `outputs/report02_derived.json:photo`", "공격": "IoU 0.875 ⟨outputs/report02_derived.json : photo.best_iou⟩ 는 눈금 없는 숫자다 — 형상 정확도의 어떤 기준에 대는 값인가", "답": "같은 메쉬로 만든 가짜 사진을 같은 파이프라인에 넣은 자기복제 상한 0.864 ⟨outputs/report02_derived.json : photo.ceiling_min⟩~0.957 ⟨outputs/report02_derived.json : photo.ceiling_max⟩ 을 함께 싣고 상한 대비로 읽는다. 자세 1° 오차에서 그 지표가 0.906 ⟨outputs/report02_derived.json : photo.iou_at_1deg_pose_error⟩ 로 내려간다"}, {"주장": "이 편의 기여는 GPU 광선엔진 위의 부품별 재질 PO 와 그것을 검출 사슬까지 잇는 파이프라인 통합이다", "근거": "§2 표 · 01편 §3 · `outputs/prior_settled_h8.json`", "공격": "Rzewuski(NATO STO 2021)가 FDTD 로 드론 바이스태틱 RCS 를 패시브 예산에 넣고 50 m OTA 검출까지 냈다 — 같은 산출물이다", "답": "같은 산출물을 다른 엔진으로 낸다는 것을 그대로 적는다. 우리가 더하는 것은 GPU 광선엔진 안에서의 부품별 재질 PO, 교정된 Pfa 위의 세 파형 통제 비교, 그리고 σ 오차 아래 순위 강건성의 수치화다(§5). 게재본과 프리프린트 구분은 01편 §3 에 있다"}]}-->
## §7. 방어선

| 주장 | 근거 | 공격 | 답 |
|---|---|---|---|
| 표적 σ 는 부품별 재질 메쉬 위의 PO 면적분으로 계산하고, 커널은 해석 PO 기준해와 kr 1~100 전 구간에서 0.201 dB ⟨outputs/sbr_kr_sweep.json : summary_div16.max_abs_db_vs_po⟩ 안에서 맞는다 | 그림 5 · 표 §3.1 · `outputs/sbr_kr_sweep.json:summary_div16.max_abs_db_vs_po` | PO 는 few-λ 표적에서 부정확하다 — 드론이 바로 그 크기다 | PO 모델 자체의 간극을 정확 Mie 기준해로 따로 쟀다 — kr ≥ 9.06 ⟨outputs/report02_derived.json : po_floor.kr_below_1p0_db⟩ 에서 1 dB, kr ≥ 15.16 ⟨outputs/report02_derived.json : po_floor.kr_below_0p5_db⟩ 에서 0.5 dB 안이다 ⟨outputs/report02_derived.json : po_floor⟩. 21 ⟨outputs/report02_derived.json : electrical.n_airframe_band⟩ 조합 중 1 ⟨outputs/report02_derived.json : electrical.n_below_po_1db⟩ 개가 그 문턱 아래이고, 그 자리를 §4 앵커가 잡는다 |
| 가림 판정은 Sionna 의 Mitsuba/OptiX 광선엔진이 하고, 표면전류 적분과 σ 출력을 우리가 얹었다 | §2 표 · `outputs/prior_settled_sionna.json:word_counts_rerun_this_session` | Sionna 에도 SBR 이 있으니 엔진 기여는 이미 그 안에 있다 | 기술보고서(v1.2, 59쪽)에 SBR 은 48 ⟨outputs/prior_settled_sionna.json : word_counts_rerun_this_session.sionna_rt_technical_report_v2_59p.SBR or shooting-and-bouncing⟩회 나오고 우리도 그 엔진을 그대로 쓴다. 같은 문서에서 `physical optics` 0 ⟨outputs/prior_settled_sionna.json : word_counts_rerun_this_session.sionna_rt_technical_report_v2_59p.physical optics⟩회 · `radar cross section` 0 ⟨outputs/prior_settled_sionna.json : word_counts_rerun_this_session.sionna_rt_technical_report_v2_59p.radar cross section⟩회이므로, 더한 것은 표면적분과 σ 출력이다 |
| 절대 레벨은 우리 PO 출력이고, 측정에서 받은 것은 주파수 기울기 하나다 | 그림 6 · 표 §4.2 · `outputs/report02_derived.json:anchor_modes` | 절대 σ 가 측정으로 검증되지 않았다면 검출 결과 전체가 흔들린다 | 공통모드 σ 오차는 세 밴드를 함께 옮겨 15 ⟨outputs/report02_derived.json : sigma_sens.n_cells⟩셀 전부에서 순위를 그대로 두고, 절대거리만 σ 1 dB 당 0.246 dB ⟨outputs/report02_derived.json : sigma_sens.slope_mean_db_per_db⟩ 움직인다 ⟨outputs/sigma_sensitivity.json : common_mode⟩. 논문은 순위를 주장하고, 절대 레벨은 06편 교정구로 앵커한다 |
| 밴드별 차분 σ 오차의 뒤집힘 문턱은 기체별 0.61 ⟨outputs/report02_derived.json : sigma_sens.flip_single_min_db⟩~7.42 dB ⟨outputs/report02_derived.json : sigma_sens.flip_single_max_db⟩ 다 | 그림 7 · 표 §5 · `outputs/sigma_sensitivity.json:differential` | 그 문턱이 현실 차분 폭 5.01 dB ⟨outputs/report02_derived.json : sigma_sens.realistic_span_db⟩ 안에 드는 기체가 셋이다 — 그 폭 안에서 순위가 바뀐다 | 인용 규약을 두 개로 고정해서 답한다 — 자세평균 σ 로 인용하면 다섯 기체가 한 순위에 합의하고, 측정 기울기를 얹으면 최악 문턱이 0.09 ⟨outputs/report02_derived.json : sigma_sens.worst_flip_aspect_avg_db⟩→3.72 dB ⟨outputs/report02_derived.json : sigma_sens.worst_flip_anchored_db⟩ 로 올라간다. 남은 차분은 06편이 두 기체의 밴드별 상대 레벨로 닫는다 |
| 바이스태틱 자세 패턴은 β ≤ 45° 에서 성립하고, 그 범위의 상반성 RMS 는 2.57 dB ⟨outputs/sbr_defect_fixes.json : d2_exit_vis_effect_on_reciprocity.rms_with_exit_vis_db[3]⟩ 다 | §2.1 · `outputs/sbr_defect_fixes.json:d2_exit_vis_effect_on_reciprocity` | β > 45° 에서 상반성이 크게 깨진다면 엔진 자체를 믿기 어렵다 | 출사 쪽 가림을 켜면 위반 최대치가 9.69 ⟨outputs/sbr_defect_fixes.json : d2_exit_vis_effect_on_reciprocity.worst_without_exit_vis_db⟩ → 8.24 dB ⟨outputs/sbr_defect_fixes.json : d2_exit_vis_effect_on_reciprocity.worst_with_exit_vis_db⟩ 로 내려간다. 논문은 β ≤ 45° 만 쓰고, 그 위는 2회 반사를 β 별로 다시 돌려 §6 에서 넓힌다 |
| 가림은 방위평균 σ 를 기체별 0.11 ⟨outputs/report02_derived.json : occlusion.min_db⟩~6.63 dB ⟨outputs/report02_derived.json : occlusion.max_db⟩ 옮긴다 | 그림 4 · 표 §2 · `outputs/report02_derived.json:occlusion.rows` | 그 차이가 이산화 잡음일 수 있다 | PO 를 λ/7↔λ/12 로 돌린 이산화 바닥이 최대 0.071 dB ⟨outputs/report02_derived.json : occlusion.floor_max_db⟩ 이고, 7 ⟨outputs/report02_derived.json : occlusion.n_above_floor⟩기체 전부에서 가림이 그 위에 있다 |
| 메쉬 형상은 사진 실루엣 · 제조사 CAD 치수 · 실물 유래 메쉬 셋으로 검사했다 | 그림 2 · 표 §1.2 · `outputs/report02_derived.json:photo` | IoU 0.875 ⟨outputs/report02_derived.json : photo.best_iou⟩ 는 눈금 없는 숫자다 — 형상 정확도의 어떤 기준에 대는 값인가 | 같은 메쉬로 만든 가짜 사진을 같은 파이프라인에 넣은 자기복제 상한 0.864 ⟨outputs/report02_derived.json : photo.ceiling_min⟩~0.957 ⟨outputs/report02_derived.json : photo.ceiling_max⟩ 을 함께 싣고 상한 대비로 읽는다. 자세 1° 오차에서 그 지표가 0.906 ⟨outputs/report02_derived.json : photo.iou_at_1deg_pose_error⟩ 로 내려간다 |
| 이 편의 기여는 GPU 광선엔진 위의 부품별 재질 PO 와 그것을 검출 사슬까지 잇는 파이프라인 통합이다 | §2 표 · 01편 §3 · `outputs/prior_settled_h8.json` | Rzewuski(NATO STO 2021)가 FDTD 로 드론 바이스태틱 RCS 를 패시브 예산에 넣고 50 m OTA 검출까지 냈다 — 같은 산출물이다 | 같은 산출물을 다른 엔진으로 낸다는 것을 그대로 적는다. 우리가 더하는 것은 GPU 광선엔진 안에서의 부품별 재질 PO, 교정된 Pfa 위의 세 파형 통제 비교, 그리고 σ 오차 아래 순위 강건성의 수치화다(§5). 게재본과 프리프린트 구분은 01편 §3 에 있다 |

### §7. 인용

1. Das et al., "Multiband Monostatic and Bistatic RCS Characterization of AAVs for ISAC Channel Modeling", IEEE Wireless Communications Letters 15:3731-3735, 2026 [게재] doi:10.1109/LWC.2026.3705634 (Table III · 기울기 앵커(§4))<!--pk:cite {"kind": "cite", "authors": "Das et al.", "title": "Multiband Monostatic and Bistatic RCS Characterization of AAVs for ISAC Channel Modeling", "venue": "IEEE Wireless Communications Letters", "volume": "15", "pages": "3731-3735", "year": 2026, "status": "published", "status_ko": "게재", "arxiv": null, "doi": "10.1109/LWC.2026.3705634", "note": "Table III · 기울기 앵커(§4)", "text": "Das et al., \"Multiband Monostatic and Bistatic RCS Characterization of AAVs for ISAC Channel Modeling\", IEEE Wireless Communications Letters 15:3731-3735, 2026 [게재] doi:10.1109/LWC.2026.3705634 (Table III · 기울기 앵커(§4))"}-->
2. Hoydis et al., "Sionna RT Technical Report", NVIDIA technical report (document version 1.2), 2025 [프리프린트, arXiv:2504.21719] (SBR 48회 · PO 표면적분 0회(§2))<!--pk:cite {"kind": "cite", "authors": "Hoydis et al.", "title": "Sionna RT Technical Report", "venue": "NVIDIA technical report (document version 1.2)", "volume": null, "pages": null, "year": 2025, "status": "preprint", "status_ko": "프리프린트", "arxiv": "2504.21719", "doi": null, "note": "SBR 48회 · PO 표면적분 0회(§2)", "text": "Hoydis et al., \"Sionna RT Technical Report\", NVIDIA technical report (document version 1.2), 2025 [프리프린트, arXiv:2504.21719] (SBR 48회 · PO 표면적분 0회(§2))"}-->
3. Liu et al., "Clutter-Aware Integrated Sensing and Communication: Models, Methods, and Future Directions", Proceedings of the IEEE 114(1):52-91, 2026 [게재] doi:10.1109/JPROC.2026.3675476 (드론 메쉬를 Sionna 에 넣고 stock 응답을 받는 게재 선행(§2))<!--pk:cite {"kind": "cite", "authors": "Liu et al.", "title": "Clutter-Aware Integrated Sensing and Communication: Models, Methods, and Future Directions", "venue": "Proceedings of the IEEE", "volume": "114(1)", "pages": "52-91", "year": 2026, "status": "published", "status_ko": "게재", "arxiv": null, "doi": "10.1109/JPROC.2026.3675476", "note": "드론 메쉬를 Sionna 에 넣고 stock 응답을 받는 게재 선행(§2)", "text": "Liu et al., \"Clutter-Aware Integrated Sensing and Communication: Models, Methods, and Future Directions\", Proceedings of the IEEE 114(1):52-91, 2026 [게재] doi:10.1109/JPROC.2026.3675476 (드론 메쉬를 Sionna 에 넣고 stock 응답을 받는 게재 선행(§2))"}-->
4. Zhang et al., "A Unified RCS Modeling of Typical Targets for 3GPP ISAC Channel Standardization", IEEE Journal on Selected Areas in Communications 44:702-716, 2026 [게재] doi:10.1109/JSAC.2025.3608732 (sigma = A(f)·B1(phi,theta)·B2 분해의 출처(§4))<!--pk:cite {"kind": "cite", "authors": "Zhang et al.", "title": "A Unified RCS Modeling of Typical Targets for 3GPP ISAC Channel Standardization", "venue": "IEEE Journal on Selected Areas in Communications", "volume": "44", "pages": "702-716", "year": 2026, "status": "published", "status_ko": "게재", "arxiv": null, "doi": "10.1109/JSAC.2025.3608732", "note": "sigma = A(f)·B1(phi,theta)·B2 분해의 출처(§4)", "text": "Zhang et al., \"A Unified RCS Modeling of Typical Targets for 3GPP ISAC Channel Standardization\", IEEE Journal on Selected Areas in Communications 44:702-716, 2026 [게재] doi:10.1109/JSAC.2025.3608732 (sigma = A(f)·B1(phi,theta)·B2 분해의 출처(§4))"}-->
5. Rzewuski, Kulpa, Pachwicewicz, Malanowski, Salski, "Drone Detectability Feasibility Study using Passive Radars Operating in WIFI and DVB-T Band", NATO STO-MP-MSG-SET-183, paper 13, 2021 [게재] (FDTD 로 같은 산출물 — 신규성은 엔진과 통합에 있다)<!--pk:cite {"kind": "cite", "authors": "Rzewuski, Kulpa, Pachwicewicz, Malanowski, Salski", "title": "Drone Detectability Feasibility Study using Passive Radars Operating in WIFI and DVB-T Band", "venue": "NATO STO-MP-MSG-SET-183, paper 13", "volume": null, "pages": null, "year": 2021, "status": "published", "status_ko": "게재", "arxiv": null, "doi": null, "note": "FDTD 로 같은 산출물 — 신규성은 엔진과 통합에 있다", "text": "Rzewuski, Kulpa, Pachwicewicz, Malanowski, Salski, \"Drone Detectability Feasibility Study using Passive Radars Operating in WIFI and DVB-T Band\", NATO STO-MP-MSG-SET-183, paper 13, 2021 [게재] (FDTD 로 같은 산출물 — 신규성은 엔진과 통합에 있다)"}-->
6. Pasquale et al., "BVH-Accelerated Ray Tracing for High-Frequency Electromagnetic Backscattering", arXiv preprint, 2026 [프리프린트, arXiv:2604.09243] (바이스태틱 출사 가림을 지목한 각주(§2.1))<!--pk:cite {"kind": "cite", "authors": "Pasquale et al.", "title": "BVH-Accelerated Ray Tracing for High-Frequency Electromagnetic Backscattering", "venue": "arXiv preprint", "volume": null, "pages": null, "year": 2026, "status": "preprint", "status_ko": "프리프린트", "arxiv": "2604.09243", "doi": null, "note": "바이스태틱 출사 가림을 지목한 각주(§2.1)", "text": "Pasquale et al., \"BVH-Accelerated Ray Tracing for High-Frequency Electromagnetic Backscattering\", arXiv preprint, 2026 [프리프린트, arXiv:2604.09243] (바이스태틱 출사 가림을 지목한 각주(§2.1))"}-->